# Tugas 2

## 1. Install library terlebih dahulu

In [2]:
pip install pandas openpyxl numpy scikit-learn

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 1.9 MB/s eta 0:00:05
   ----- ---------------------------------- 1.0/8.3 MB 1.7 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/8.3 MB 1.9 MB/s eta 0:00:04
   -------- ------------------------------- 1.8/8.3 MB 1.8 MB/s eta 0:00:04
   ----------- ---------------------------- 2.4/8.3 MB 2.0 MB/s eta 0:00:04
   ------------- -------------------------- 2.9/8.3 MB 2.0 MB/s eta 0:00:03
   --------------- ------------------------ 3.1/8.3 MB 1.9 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.3 MB 1.9 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.3 MB 1.8 MB/s eta 0:00:03
   ------------------ ----------------

## 2. Import library

In [1]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 3. Membaca dataset

In [3]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

df.head()

,id,isi_berita,label
0,1,Marco Bezzecchi tak sabar menghadapi MotoGP Sa...,sport
1,2,Marc Marquez masih harus beradaptasi dengan ko...,sport
2,3,Pebalap Mercedes GP Kimi Antonelli tampil luar...,sport
3,4,Selesai sudah Seri IV M-15 Men's World Tennis ...,sport
4,5,Jete Run Festival 2026 baru saja selesai digel...,sport


### Cek jumlah data:

In [4]:
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label'], dtype='str')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


## 4. Mengubah label menjadi numerik

In [5]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [6]:
print(df["label_num"].value_counts())

label_num
1    100
0    100
Name: count, dtype: int64


## 5. Menghitung jumlah kata semua berita

In [7]:
df["jumlah_kata_asli"] = df["isi_berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["id", "jumlah_kata_asli"]].head()

total_kata = df["jumlah_kata_asli"].sum()

print("Total seluruh kata :", total_kata)

print(df["jumlah_kata_asli"].describe())

Total seluruh kata : 59951
count     200.000000
mean      299.755000
std       137.545369
min        43.000000
25%       217.000000
50%       274.000000
75%       352.250000
max      1085.000000
Name: jumlah_kata_asli, dtype: float64


### melihat jumlah kata setiap berita

In [8]:
df[[
    "id",
    "label",
    "jumlah_kata_asli"
]]

,id,label,jumlah_kata_asli
0,1,sport,256
1,2,sport,243
2,3,sport,224
3,4,sport,268
4,5,sport,509
...,...,...,...
195,196,finance,148
196,197,finance,153
197,198,finance,278
198,199,finance,211


## 6. Kamus kata tidak baku

In [9]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

## 7. Kamus bahasa asing

In [10]:
kamus_asing = {
    
    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",
    
    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

## 8. Fungsi preprocessing

In [11]:
def preprocessing(text):
    
    # Pastikan berupa string
    text = str(text)
    
    # =========================
    # CASE FOLDING
    # =========================
    text = text.lower()
    
    
    # =========================
    # HAPUS URL
    # =========================
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )
    
    
    # =========================
    # HAPUS EMAIL
    # =========================
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )
    
    
    # =========================
    # HAPUS ANGKA
    # =========================
    text = re.sub(
        r'\d+',
        ' ',
        text
    )
    
    
    # =========================
    # HAPUS TANDA BACA,
    # SIMBOL DAN EMOTICON
    # =========================
    
    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )
    
    
    # =========================
    # HAPUS SPASI BERLEBIH
    # =========================
    
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()
    
    
    # =========================
    # TOKENISASI
    # =========================
    
    kata = text.split()
    
    
    hasil = []
    
    for token in kata:
        
        # Membakukan kata
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]
        
        
        # Bahasa asing -> Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]
        
        
        hasil.append(token)
    
    
    return " ".join(hasil)

## 9. Terapkan preprocessing ke semua berita

In [12]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[[
    "id",
    "isi_berita",
    "berita_clean"
]].head()

,id,isi_berita,berita_clean
0,1,Marco Bezzecchi tak sabar menghadapi MotoGP Sa...,marco bezzecchi tak sabar menghadapi motogp sa...
1,2,Marc Marquez masih harus beradaptasi dengan ko...,marc marquez masih harus beradaptasi dengan ko...
2,3,Pebalap Mercedes GP Kimi Antonelli tampil luar...,pebalap mercedes gp kimi antonelli tampil luar...
3,4,Selesai sudah Seri IV M-15 Men's World Tennis ...,selesai sudah seri iv m men s world tennis kej...
4,5,Jete Run Festival 2026 baru saja selesai digel...,jete run festival baru saja selesai digelar ak...


## 10. Hitung jumlah kata setelah preprocessing

In [13]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

df[[
    "id",
    "jumlah_kata_asli",
    "jumlah_kata_clean"
]].head()

,id,jumlah_kata_asli,jumlah_kata_clean
0,1,256,252
1,2,243,243
2,3,224,215
3,4,268,257
4,5,509,505


### Total:

In [14]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 59951
Total kata setelah preprocessing : 58702


## 11. Ekstrak seluruh kata unik

In [15]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 58702
Jumlah kata unik: 6631


In [16]:
kata_unik[:100]

['a',
 'aan',
 'abdi',
 'abdul',
 'abimanyu',
 'absen',
 'absennya',
 'absorber',
 'abu',
 'abullah',
 'acara',
 'access',
 'account',
 'acd',
 'aceh',
 'acosta',
 'activ',
 'activation',
 'activities',
 'acuan',
 'ada',
 'adain',
 'adalah',
 'adanya',
 'adaptif',
 'adapun',
 'ade',
 'adhitya',
 'adi',
 'adik',
 'adil',
 'adisutjipto',
 'administrasi',
 'adna',
 'adopsi',
 'adp',
 'ads',
 'adu',
 'aduan',
 'advisory',
 'aerodinamika',
 'aerodrome',
 'aeronautika',
 'af',
 'aff',
 'affected',
 'afpi',
 'afrianto',
 'ag',
 'agama',
 'agar',
 'agen',
 'agenda',
 'agent',
 'agoeng',
 'agraria',
 'agreement',
 'agregat',
 'agresif',
 'agresivitas',
 'agrinas',
 'agu',
 'agunan',
 'agung',
 'agus',
 'agusman',
 'agustian',
 'agustus',
 'ahi',
 'ahmad',
 'ahren',
 'ahsanurrohim',
 'ahy',
 'ai',
 'aichi',
 'air',
 'airbus',
 'airlangga',
 'airline',
 'airlines',
 'airmen',
 'airnav',
 'airport',
 'airports',
 'ajaib',
 'ajakan',
 'ajang',
 'ajir',
 'ajukan',
 'akademisi',
 'akal',
 'akan',
 'a

In [17]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,a
1,aan
2,abdi
3,abdul
4,abimanyu
5,absen
6,absennya
7,absorber
8,abu
9,abullah


In [18]:
df_kata_unik.to_excel(
    "kata_unik.xlsx",
    index=False
)

## 12. Membagi data 160 training dan 40 testing

In [19]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    
    df["berita_clean"],
    df["label_num"],
    
    test_size=40,
    
    random_state=42,
    
    stratify=df["label_num"]
)

In [20]:
print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


In [21]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 13. Representasi TF-IDF

In [22]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [23]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 2856


## 14. Melihat matriks TF-IDF

In [24]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,activation,ada,adalah,...,youtube,yoy,yudhi,yudhoyono,zapp,zarco,zona,zulhas,zulkifli,zulverdi
0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.032134,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.030565,0.0,0.057961,0.0,0.0,0.0,0.018835,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.030240,0.071113,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,activation,ada,adalah,...,yoy,yudhi,yudhoyono,zapp,zarco,zona,zulhas,zulkifli,zulverdi,label
0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.032134,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.030565,0.0,0.057961,0.0,0.0,0.0,0.018835,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.030240,0.071113,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


## 15. Simpan data TF-IDF

In [26]:
tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

In [27]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

In [28]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [29]:
for kelas in sorted(
    y_train.unique()
):
    
    mask = (
        y_train_array == kelas
    )
    
    X_class = X_train_tfidf[
        mask
    ]
    
    
    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata
    
    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )
    
    
    jumlah_dokumen_class = (
        X_class.shape[0]
    )
    
    
    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )
    
    
    class_space_density += (
        class_density
    )

In [30]:
epsilon = 1e-12

icsdf = np.log(
    
    (jumlah_kelas + epsilon)
    
    /
    
    (class_space_density + epsilon)
)

In [31]:
df_icsdf = pd.DataFrame({
    
    "kata": features,
    
    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
2855,zulverdi,4.382027
2832,xi,4.382027
2831,wujud,4.382027
2829,work,4.382027
2828,won,4.382027
2826,wilayahnya,4.382027
2824,wihi,4.382027
2822,white,4.382027
2821,wec,4.382027
2818,wartawan,4.382027


## 17. TF-IDF × ICSDF

In [32]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [33]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

## 18. Menentukan kata paling penting

In [34]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [35]:
TOP_K = 100

In [36]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [37]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['bandara', 'marquez', 'bilal', 'balapan', 'marc', 'beras',
       'purbaya', 'motogp', 'bezzecchi', 'penerbangan', 'set', 'artikel',
       'olahraga', 'saham', 'aragon', 'martin', 'rp', 'penumpang', 'saya',
       'asian', 'pertandingan', 'pensiun', 'ufc', 'aset', 'pukul',
       'vulkanik', 'poin', 'wib', 'umar', 'reza'], dtype=object)

In [38]:
df_kata_penting = pd.DataFrame({
    
    "kata": kata_penting,
    
    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,bandara,0.091810
1,marquez,0.066488
2,bilal,0.053062
3,balapan,0.052861
4,marc,0.052179
5,beras,0.049791
6,purbaya,0.046822
7,motogp,0.046587
8,bezzecchi,0.043914
9,penerbangan,0.043832


## 19. Reduksi menjadi 100 fitur ICSDF

In [39]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

In [40]:
print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 2856)
Sesudah ICSDF: (160, 100)


## 20. Tabel hasil ICSDF

In [41]:
icsdf_train_df = pd.DataFrame(
    
    X_train_selected.toarray(),
    
    columns=kata_penting
)

icsdf_train_df["label"] = (
    
    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,bandara,marquez,bilal,balapan,marc,beras,purbaya,motogp,bezzecchi,penerbangan,...,motor,finis,juta,masing,kemenangan,emas,karhutla,month,negara,label
0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0
1,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1
2,1.053046,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.611119,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0
3,0.000000,0.082596,0.0,0.366097,0.082596,0.0,0.0,0.277527,0.964288,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1
4,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.235421,0.0,0.0,0.0,0.0,1


In [42]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

## 21. PCA

In [43]:
pca = PCA(
    
    n_components=20,
    
    random_state=42
)

In [44]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [45]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [46]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


## 22. Membuat tabel data reduksi training

In [47]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [48]:
df_train_reduksi = pd.DataFrame(
    
    X_train_pca,
    
    columns=nama_pc
)

df_train_reduksi["label"] = (
    
    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.001121,-0.006913,0.160983,-0.002394,-0.094889,-0.026122,-0.049728,-0.111334,0.067785,-0.038615,...,0.160285,-0.277782,-0.363865,0.528277,0.018553,0.161674,0.038191,-0.093984,-0.115068,0
1,-0.027303,-0.019351,0.103181,-0.004465,-0.054495,0.012588,-0.031202,-0.039032,-0.090246,-0.024768,...,0.050273,-0.053849,-0.087261,-0.073120,-0.066778,-0.105760,-0.024867,-0.015009,-0.013821,1
2,1.143420,0.214979,-0.545417,0.012365,0.183723,-0.036122,0.207861,-0.063544,-0.001715,-0.042386,...,-0.015550,0.034894,0.066012,0.086462,0.123476,-0.152388,0.007781,-0.010151,-0.015383,0
3,-0.302511,-0.352149,-0.337805,0.000011,0.055014,0.070834,0.044910,0.064016,0.061408,0.014028,...,-0.024732,0.024326,0.025264,0.022478,0.028253,0.014122,0.030334,-0.056796,0.054718,1
4,-0.038198,0.001923,0.104211,-0.037229,-0.095635,0.296299,0.152815,0.008628,0.003049,0.002985,...,0.026408,-0.024133,-0.033361,-0.022943,-0.019421,-0.019419,-0.011918,0.014002,-0.004926,1


## 23. Data testing

In [49]:
df_test_reduksi = pd.DataFrame(
    
    X_test_pca,
    
    columns=nama_pc
)

df_test_reduksi["label"] = (
    
    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.027229,-0.017200,0.499216,-0.017190,0.848524,0.031583,0.048039,0.086194,-0.014870,0.049790,...,-0.063920,-0.038847,0.004428,-0.014817,-0.002288,-0.031540,0.000412,0.003492,-0.000681,0
1,-0.107044,-0.140222,0.003526,0.003732,0.001521,-0.035759,-0.009848,-0.061092,0.029452,0.036999,...,0.118706,0.049778,-0.001988,-0.030637,-0.054013,0.020298,0.004450,0.018808,-0.022860,1
2,-0.054097,0.029645,0.059969,0.003899,-0.051407,0.019169,-0.032084,-0.043313,-0.099585,-0.022612,...,0.040951,-0.031954,-0.069147,-0.064813,-0.072381,-0.096404,-0.008469,-0.039539,-0.011282,1
3,-0.892249,1.870718,-0.581279,-0.962246,0.165229,-0.191196,-0.047935,0.044877,0.060317,0.002614,...,-0.037907,0.035757,0.048539,0.030334,0.029768,0.031421,-0.005359,-0.018817,0.007712,1
4,-0.010700,-0.010083,0.290290,-0.002000,-0.144750,-0.138704,0.016755,-0.202558,0.020873,0.505218,...,-0.054603,0.217607,0.079243,0.057048,0.074259,0.026659,0.017129,-0.019862,-0.037342,0


## 24. Simpan data reduksi

In [50]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

## 25. Cek total training dan testing

In [51]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


## 26. Kolom terakhir harus label

In [52]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'label'],
      dtype='str')